# Clustering with k-Means

Clustering is **unsupervised**: k-Means makes groups from the features only. It **does not look at the labels**.
After that, we compare the groups with the true labels. This tells us how the classes are placed in the feature space.

Questions we answer:
* Do the sentiment examples form 3 natural groups (Negative, Neutral, Positive)?
* Do images of the same class end up in the same cluster? Which classes are grouped together?
* What happens when we change the number of clusters k?

k-Means is written by us (NumPy only). Only the scores ARI and NMI come from `sklearn.metrics` (metrics are allowed).

## 1. Setup

In [ ]:
import os
import sys

# The notebook is in a sub-folder. Go to the project folder (where common.py is).
if not os.path.exists("common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score   # metrics only

import common   # our shared file: load data, split, metrics, Kaggle file

common.set_seed(42)
os.makedirs("clustering/outputs", exist_ok=True)

## 2. Load the data

No train/validation split is needed: we cluster **all** training examples (20,000 images, 12,000 sentences).
The labels are only used at the end, to check the clusters.

k-Means uses distances, so we use the same preprocessing as k-NN:
* sentiment → `standardize`
* image → `sqrt` (square root, then every row has length 1)

In [ ]:
X_image_raw, y_image, _, _ = common.load_task("image")
X_sent_raw, y_sent, _, _ = common.load_task("sentiment")


def preprocess(X, mode):
    stats = common.fit_preprocessing(X, mode)
    return common.apply_preprocessing(X, mode, stats)


X_sent = preprocess(X_sent_raw, "standardize")
X_image = preprocess(X_image_raw, "sqrt")
print("sentiment:", X_sent.shape, "| image:", X_image.shape)

## 3. k-Means

1. **Start:** choose k starting centres.
2. **Assign:** put every example in the cluster of its closest centre.
3. **Update:** move every centre to the mean of its examples.
4. Repeat 2 and 3 until the centres stop moving.

**Starting centres:**
* `random` – k random examples.
* `k-means++` – the first centre is random. Each next centre is chosen at random, but examples **far** from the centres we already have are more likely. So the centres start spread out. This usually gives better clusters.

The result depends on the start, so we run k-Means 3 times and keep the best run.
The best run has the smallest **inertia** = sum of squared distances from each example to its centre (smaller = tighter clusters).

In [ ]:
def squared_distances(X, centers):
    """Squared distance from every example to every centre. Uses |x - c|^2 = |x|^2 - 2 x·c + |c|^2."""
    return (X ** 2).sum(axis=1)[:, None] - 2 * X @ centers.T + (centers ** 2).sum(axis=1)[None, :]


def kmeans_plus_plus_start(X, k, rng):
    centers = [X[rng.integers(len(X))]]                     # first centre: a random example
    closest = ((X - centers[0]) ** 2).sum(axis=1)           # squared distance to the nearest centre so far
    for _ in range(1, k):
        probabilities = closest / closest.sum()             # far examples are more likely
        new_center = X[rng.choice(len(X), p=probabilities)]
        centers.append(new_center)
        closest = np.minimum(closest, ((X - new_center) ** 2).sum(axis=1))
    return np.array(centers)


def kmeans_once(X, k, seed, init="k-means++", max_iter=100):
    rng = np.random.default_rng(seed)
    if init == "random":                                                 # 1. start
        centers = X[rng.choice(len(X), size=k, replace=False)].copy()
    else:
        centers = kmeans_plus_plus_start(X, k, rng)

    for iteration in range(max_iter):
        labels = squared_distances(X, centers).argmin(axis=1)            # 2. assign

        new_centers = np.zeros_like(centers)                             # 3. update
        for c in range(k):
            members = X[labels == c]
            if len(members) > 0:
                new_centers[c] = members.mean(axis=0)
            else:
                new_centers[c] = X[rng.integers(len(X))]                 # empty cluster: random example

        moved = np.abs(new_centers - centers).max()
        centers = new_centers
        if moved < 1e-6:                                                 # 4. centres stopped moving
            break

    D = squared_distances(X, centers)
    labels = D.argmin(axis=1)
    inertia = D[np.arange(len(X)), labels].sum()
    return labels, centers, inertia


def kmeans(X, k, init="k-means++", n_runs=3):
    best = None
    for seed in range(n_runs):
        labels, centers, inertia = kmeans_once(X, k, seed, init)
        if best is None or inertia < best[2]:
            best = (labels, centers, inertia)
    return best

## 4. How we score the clusters

* **Table:** row = cluster, column = true class, cell = how many examples.
* **Purity:** give each cluster the label of its biggest class, then count the correct ones. 1.0 = every cluster has only one class. Purity always goes up when k goes up, so only compare purity for the same k.
* **ARI** (Adjusted Rand Index): 1 = perfect match, about 0 = like random groups.
* **NMI** (Normalised Mutual Information): 0 = the clusters tell nothing about the labels, 1 = perfect.

In [ ]:
def cluster_table(cluster_labels, true_labels, k, n_classes):
    table = np.zeros((k, n_classes), dtype=int)
    for c, t in zip(cluster_labels, true_labels):
        table[c][t] += 1
    return table


def purity(table):
    return table.max(axis=1).sum() / table.sum()        # biggest class of each cluster


results = []


def run(task, X, y, k, note="", init="k-means++"):
    start = time.time()
    labels, centers, inertia = kmeans(X, k, init)
    table = cluster_table(labels, y, k, common.N_CLASSES[task])
    row = {"task": task, "k": k, "note": note, "inertia": round(float(inertia), 1),
           "purity": round(purity(table), 4),
           "ARI": round(adjusted_rand_score(y, labels), 4),
           "NMI": round(normalized_mutual_info_score(y, labels), 4),
           "seconds": round(time.time() - start, 1)}
    results.append(row)
    print(f"{task:<9} k={k:<4} {note:<13} purity {row['purity']:.3f}   ARI {row['ARI']:.3f}"
          f"   NMI {row['NMI']:.3f}   ({row['seconds']}s)")
    return labels, table

## 5. Sentiment

### 5.1 Three clusters: do they match Negative / Neutral / Positive?

If they did, each cluster would be mostly one sentiment.

In [ ]:
sent_clusters, table = run("sentiment", X_sent, y_sent, 3)

shares = table / table.sum(axis=1, keepdims=True)       # share of each sentiment inside each cluster
display(pd.DataFrame(table, columns=common.SENTIMENT_NAMES, index=[f"cluster {c}" for c in range(3)]))

plt.figure(figsize=(5, 3.5))
plt.imshow(shares, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="share of the cluster")
plt.xticks(range(3), common.SENTIMENT_NAMES)
plt.yticks(range(3), [f"cluster {c}" for c in range(3)])
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{shares[i, j]:.0%}", ha="center", va="center")
plt.title("Sentiment: what is inside each cluster")
plt.show()

If every cluster has about 1/3 of each sentiment, the main structure of the BERT features is **not** sentiment (maybe topic or style). This also explains why sentiment is hard to classify.

### 5.2 Different numbers of clusters

In [ ]:
for k in [2, 3, 4, 5, 6, 8, 10]:
    run("sentiment", X_sent, y_sent, k)

## 6. Image

### 6.1 Fifty clusters (one per class)

Two quick checks first: raw features vs `sqrt` features, and random start vs k-means++.

In [ ]:
run("image", preprocess(X_image_raw, "none"), y_image, 50, note="raw features")
run("image", X_image, y_image, 50, note="random start", init="random")
image_clusters, image_table = run("image", X_image, y_image, 50, note="sqrt")

### 6.2 Which classes are grouped together?

For every class we find its **main cluster** (the cluster with most of its images).
Classes with the same main cluster look alike to k-Means. Compare them with the classes that k-NN and the neural network confuse.

In [ ]:
main_cluster = image_table.argmax(axis=0)                      # for every class: its main cluster
share_in_main = image_table.max(axis=0) / image_table.sum(axis=0)

print("Clusters that are the main cluster of more than one class:")
for cluster in range(50):
    classes = [c for c in range(50) if main_cluster[c] == cluster]
    if len(classes) > 1:
        print(f"   cluster {cluster:2d}: classes {classes}")

order = np.argsort(share_in_main)
print("\nClasses most spread over many clusters:", [(int(c), f"{share_in_main[c]:.0%}") for c in order[:5]])
print("Classes most in one cluster:", [(int(c), f"{share_in_main[c]:.0%}") for c in order[-5:]])

cluster_purity = image_table.max(axis=1) / np.maximum(image_table.sum(axis=1), 1)
plt.figure(figsize=(9, 3))
plt.bar(range(50), np.sort(cluster_purity)[::-1])
plt.xlabel("cluster (sorted)")
plt.ylabel("share of its biggest class")
plt.title("Image: how clean each cluster is")
plt.grid(axis="y", alpha=0.3)
plt.show()

### 6.3 Different numbers of clusters

In [ ]:
for k in [10, 25, 50, 75, 100]:
    run("image", X_image, y_image, k)

## 7. All results and the elbow plot

**Elbow method:** inertia always goes down when k goes up. Where the curve bends (the "elbow"), more clusters stop helping much.

In [ ]:
table = pd.DataFrame(results)
table.to_csv("clustering/outputs/clustering_results.csv", index=False)
display(table)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for row, task in enumerate(["sentiment", "image"]):
    r = table[(table["task"] == task) & (table["note"] == "")].sort_values("k")
    axes[row, 0].plot(r["k"], r["inertia"], marker="o")
    axes[row, 0].set_title(f"{task}: inertia (elbow)")
    for score in ["purity", "ARI", "NMI"]:
        axes[row, 1].plot(r["k"], r[score], marker="o", label=score)
    axes[row, 1].set_title(f"{task}: match with the true labels")
    axes[row, 1].legend()
    for ax in axes[row]:
        ax.set_xlabel("k (number of clusters)")
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 2-D pictures with PCA

The features have 500 or 768 dimensions, so we cannot plot them directly.
**PCA** finds the 2 directions where the data changes the most, and we draw every example on them.
This is only a rough view: points that overlap in 2-D can still be apart in the full space.

In [ ]:
def pca_2d(X):
    centered = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ Vt[:2].T          # project on the 2 main directions


rng = np.random.default_rng(42)
sample = rng.choice(len(X_sent), 3000, replace=False)       # 3000 points are enough to see
P = pca_2d(X_sent)[sample]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for c in range(3):
    axes[0].scatter(P[y_sent[sample] == c, 0], P[y_sent[sample] == c, 1], s=4, alpha=0.5,
                    label=common.SENTIMENT_NAMES[c])
    axes[1].scatter(P[sent_clusters[sample] == c, 0], P[sent_clusters[sample] == c, 1], s=4, alpha=0.5,
                    label=f"cluster {c}")
axes[0].set_title("Sentiment: true labels")
axes[1].set_title("Sentiment: k-Means clusters (k=3)")
for ax in axes:
    ax.legend(markerscale=4)
plt.show()

classes_to_show = [0, 1, 2, 3, 4, 5]                        # 6 classes only, so the plot is readable
mask = np.isin(y_image, classes_to_show)
P = pca_2d(X_image[mask])
plt.figure(figsize=(6, 5))
for c in classes_to_show:
    plt.scatter(P[y_image[mask] == c, 0], P[y_image[mask] == c, 1], s=5, alpha=0.5, label=f"class {c}")
plt.title("Image: 6 classes (true labels)")
plt.legend(markerscale=3)
plt.show()

## 9. Notes for the presentation

* k-Means does not use the labels. So if the clusters match the labels, the **features themselves** separate the classes.
* Low ARI / NMI for sentiment: the 3 sentiments are not separate "blobs" in the BERT space. We need a supervised model.
* Compare the image classes grouped in 6.2 with the classes that k-NN confuses most.